# Bench translation pipeline

Évaluation des algos sur le périmètre que je couvre (parsing Word/PPTX, rendering, translation/scope, translation/request). Pour la méthodo détaillée, cf. [`docs/08_methodo_translation_js.md`](../../docs/08_methodo_translation_js.md).

Ce que mesure ce notebook :

1. **parse_pdf sur le corpus client** (71 PDFs `data/`) — reprise des résultats du run complet (fichier `bench_parse_pdf_results_js.csv`).
2. **Round-trip identité Word + PPTX** — parse → render sans modif → vérifier `runs_replaced=0` et zéro perte.
3. **Translation pipeline end-to-end** — message libre → `parse_translation_request` → `apply_translation_scope` → `build_word_document`. Mesure : runs replaced/skipped, cohérence du dispatching.
4. **Tests pytest** — décompte des cas couverts.

In [1]:
import sys
!{sys.executable} -m pip install -q -e ../..

In [2]:
import subprocess
import sys
from pathlib import Path

import pandas as pd

from docpipeline.parsing.word import parse_word
from docpipeline.parsing.pptx.parse_pptx import parse_pptx
from docpipeline.rendering.word.build_document import build_word_document
from docpipeline.rendering.pptx.build_document import build_pptx_document
from docpipeline.translation.request_js import parse_translation_request
from docpipeline.translation.scope_js import apply_translation_scope, TranslationScope

ROOT      = Path('../..').resolve()
FIX       = ROOT / 'tests' / 'fixtures'
OUT       = ROOT / 'notebooks' / '_outputs' / 'bench_translation_pipeline'
OUT.mkdir(parents=True, exist_ok=True)

def section(t): print(f'\n{"="*68}\n  {t}\n{"="*68}')

# ============================================================
section('1. parse_pdf — bench corpus client (71 PDFs)')
# ============================================================
csv = ROOT / 'bench_parse_pdf_results_js.csv'
if csv.exists():
    df = pd.read_csv(csv)
    errs = df[df['error'].fillna('').astype(str).str.strip() != '']
    print(f'PDFs analysés      : {len(df)}')
    print(f'Pages totales      : {int(df["n_pages"].sum())}')
    print(f'Pages OCR-needed   : {int(df["pages_needing_ocr"].sum())}')
    print(f'Temps total        : {df["parse_seconds"].sum():.0f}s ({df["parse_seconds"].sum()/60:.1f} min)')
    print(f'Erreurs            : {len(errs)} / {len(df)}')
    print()
    print('Distribution content_type :')
    print(df['content_type'].fillna('(error)').value_counts().to_string())
    print()
    print('Distribution source_tool (top 6) :')
    print(df['source_tool'].fillna('(error)').value_counts().head(6).to_string())
    print()
    print('Par sous-corpus :')
    by_corpus = df.groupby('corpus').agg(
        n_pdfs=('path','count'),
        n_pages=('n_pages','sum'),
        avg_seconds=('parse_seconds','mean'),
        errors=('error', lambda s: s.fillna('').astype(str).str.strip().ne('').sum()),
    ).round(1)
    print(by_corpus.to_string())
else:
    print(f'(CSV introuvable : {csv}. Lancer d\'abord `notebooks/01_document_parsing/pdf/05_bench_parse_pdf_js.ipynb`.)')

# ============================================================
section('2a. Word — round-trip identité (zéro modification)')
# ============================================================
fix_docx = FIX / 'contrat_assurance.docx'
parsed_w = parse_word(fix_docx)
span_df  = parsed_w['span_df']
out_w    = OUT / 'contrat_assurance_roundtrip_js.docx'
res_w    = build_word_document(span_df, fix_docx, out_w)
print(f'Source             : {fix_docx.name}')
print(f'Paragraphes        : {len(parsed_w["paragraph_df"])}')
print(f'Spans (runs)       : {len(span_df)} ({(~span_df["in_table"].astype(bool)).sum()} body + {span_df["in_table"].sum()} cells)')
print(f'Round-trip         : replaced={res_w["runs_replaced"]} unchanged={res_w["runs_unchanged"]} skipped={res_w["runs_skipped"]}')
print(f'  → attendu : replaced=0, unchanged=26, skipped=0')
assert res_w['runs_replaced'] == 0 and res_w['runs_skipped'] == 0, 'Round-trip Word non-identité !'
print('  ✓ identity round-trip OK')

# ============================================================
section('2b. PPTX — round-trip identité')
# ============================================================
fix_pptx = FIX / 'contrat_assurance.pptx'
if fix_pptx.exists():
    parsed_p = parse_pptx(fix_pptx)
    runs_p   = parsed_p['runs_df']
    out_p    = OUT / 'contrat_assurance_roundtrip_js.pptx'
    res_p    = build_pptx_document(runs_p, fix_pptx, out_p)
    print(f'Source             : {fix_pptx.name}')
    print(f'Slides             : {len(parsed_p["slide_df"])}')
    print(f'Runs               : {len(runs_p)}')
    print(f'Round-trip         : replaced={res_p["runs_replaced"]} skipped={res_p["runs_skipped"]}')
    assert res_p['runs_replaced'] == 0, 'Round-trip PPTX non-identité !'
    print('  ✓ identity round-trip OK')
else:
    print(f'(fixture {fix_pptx.name} absente, skip)')

# ============================================================
section('3. Translation pipeline end-to-end (request + scope + render)')
# ============================================================
msg = ("Translate this contract into formal English, skip the cover, "
       "use 'deductible' for 'franchise'.")
print(f'Message            : {msg!r}')
print()

# 3.1 Request parsing
req = parse_translation_request(msg)
print('Étape 1/3  parse_translation_request :')
print(f'  target_language  = {req.target_language}')
print(f'  source_language  = {req.source_language}')
print(f'  style            = {req.style}')
print(f'  scope            = {req.scope.model_dump() if req.scope else None}')
print(f'  glossary         = {[(g.source, g.target) for g in req.glossary_additions]}')
print()

# 3.2 Scope (avec section_breadcrumb simulé pour faire matcher 'cover')
paragraph_df = parsed_w['paragraph_df'].copy()
paragraph_df['section_breadcrumb'] = 'Body'
paragraph_df.loc[paragraph_df['paragraph_index'] == 0, 'section_breadcrumb'] = 'Cover'
sl, ss, kl, ks = apply_translation_scope(paragraph_df, span_df, req.scope)
print('Étape 2/3  apply_translation_scope (Cover simulé sur paragraph_index=0) :')
print(f'  selected lines   = {len(sl)} / {len(paragraph_df)}')
print(f'  selected spans   = {len(ss)} / {len(span_df)}')
print(f'  skipped lines    = {len(kl)}')
print(f'  skipped spans    = {len(ks)}  (cover preservé)')
print(f'  cells in selected = {ss["in_table"].sum()} / {span_df["in_table"].sum()}  (cells toujours selected)')
print()

# 3.3 Render avec faux remplacement (pas de LLM ici)
ss2 = ss.copy()
ss2['text'] = ss2['text'].apply(lambda t: f'[EN] {t}' if t.strip() else t)
out_e2e = OUT / 'contrat_e2e_demo_js.docx'
res_e2e = build_word_document(ss2, fix_docx, out_e2e)
print('Étape 3/3  build_word_document :')
print(f'  runs replaced    = {res_e2e["runs_replaced"]}')
print(f'  runs unchanged   = {res_e2e["runs_unchanged"]}')
print(f'  runs skipped     = {res_e2e["runs_skipped"]}  (= 1 cover qui garde son texte source)')
print(f'  → output         : {out_e2e.relative_to(ROOT)}')

# ============================================================
section('4. Tests pytest — décompte par module')
# ============================================================
result = subprocess.run(
    [sys.executable, '-m', 'pytest', '--collect-only', '-q', str(ROOT / 'tests')],
    capture_output=True, text=True, cwd=str(ROOT)
)
lines = [l for l in result.stdout.split('\n') if l.startswith('tests/')]
by_file = {}
for l in lines:
    f = l.split('::')[0]
    by_file[f] = by_file.get(f, 0) + 1
for f, n in sorted(by_file.items()):
    print(f'  {n:>3}  {f}')
print(f'\nTotal : {sum(by_file.values())} tests collectés')

# Run effectif
result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', str(ROOT / 'tests')],
    capture_output=True, text=True, cwd=str(ROOT)
)
last = result.stdout.strip().split('\n')[-1]
print(f'Run pytest         : {last}')


  1. parse_pdf — bench corpus client (71 PDFs)
PDFs analysés      : 71
Pages totales      : 6589
Pages OCR-needed   : 22
Temps total        : 2586s (43.1 min)
Erreurs            : 0 / 71

Distribution content_type :
content_type
native    59
mixed     12

Distribution source_tool (top 6) :
source_tool
Adobe InDesign       29
Unknown              18
Microsoft Word        8
Ghostscript           7
Adobe PDF Library     2
Adobe Acrobat         2

Par sous-corpus :
                 n_pdfs  n_pages  avg_seconds  errors
corpus                                               
CG contrats MRH      11      809         26.1       0
annuel_reports       42     4424         43.3       0
cmo                   7      435         28.1       0
insurance             6      609         28.6       0
nist                  2       87          8.6       0
paper                 2       37         11.4       0
reports               1      188         71.4       0

  2a. Word — round-trip identité (zéro modific

Source             : contrat_assurance.docx
Paragraphes        : 10
Spans (runs)       : 26 (11 body + 15 cells)
Round-trip         : replaced=0 unchanged=26 skipped=0
  → attendu : replaced=0, unchanged=26, skipped=0
  ✓ identity round-trip OK

  2b. PPTX — round-trip identité
Source             : contrat_assurance.pptx
Slides             : 4
Runs               : 28
Round-trip         : replaced=0 skipped=0
  ✓ identity round-trip OK

  3. Translation pipeline end-to-end (request + scope + render)
Message            : "Translate this contract into formal English, skip the cover, use 'deductible' for 'franchise'."

Étape 1/3  parse_translation_request :
  target_language  = en
  source_language  = None
  style            = formal
  scope            = {'page_range': None, 'include_sections': None, 'exclude_sections': ['Cover']}
  glossary         = [('franchise', 'deductible')]

Étape 2/3  apply_translation_scope (Cover simulé sur paragraph_index=0) :
  selected lines   = 9 / 10
  selec

Étape 3/3  build_word_document :
  runs replaced    = 25
  runs unchanged   = 0
  runs skipped     = 1  (= 1 cover qui garde son texte source)
  → output         : notebooks\_outputs\bench_translation_pipeline\contrat_e2e_demo_js.docx

  4. Tests pytest — décompte par module


   34  tests/test_translation_request_js.py
   14  tests/test_translation_scope_js.py

Total : 48 tests collectés


Run pytest         : 48 passed in 3.21s


## Interprétation

**Section 1 (parse_pdf)** : zéro erreur sur 71 PDFs et 6589 pages confirme que le parser tient sur le corpus client réel, tous sous-corpus confondus (CG contrats MRH, annual reports, insurance, cmo, nist, paper, reports). 17% des PDFs sont en mode mixte (pages natives + scannées), ce qui justifie la stratégie `per_page_routing` plutôt qu'un choix global par document.

**Section 2 (round-trip)** : `replaced=0` sur Word et PPTX prouve que la pipeline `parse → render` est neutre quand on ne modifie rien. C'est l'invariant de base sans lequel toute traduction est suspecte (si le round-trip identité perd des runs, alors la traduction perdra aussi des runs).

**Section 3 (end-to-end)** : un message libre passe les 3 briques sans intervention manuelle. Le scope retire bien le paragraphe `Cover` (1 span skipped), garde les 15 cells du tableau (convention conservatrice tant qu'on n'a pas de section_breadcrumb par cell), et le rendering replace les 25 spans restants tout en laissant le cover intact dans la sortie.

**Section 4 (pytest)** : 48 tests verts, dont 14 pour `scope_js` (incl. cas FK cells) et 34 pour `request_js` (9 langues, 3 styles, page_range FR+EN, glossaire 3 syntaxes).

## Limites connues

- **`section_breadcrumb` non émis par parse_word/parse_pptx aujourd'hui** — d'où le `paragraph_df['section_breadcrumb'] = ...` simulé en Section 3. Quand parse_word le dérivera des Heading 1/2/3, on virera le simulacre.
- **Pas de bench LLM réel** — la cellule 3 fait un faux remplacement `[EN] {text}`. Pour mesurer la qualité de traduction (préservation styles, fidélité sémantique, glossaire respecté), il faut une clé OpenAI/Anthropic dans l'env et le Step 5 `translate_chunks` (à venir).
- **PDF span_df** — Tome 2 §0 demande un `span_df` PDF (équivalent du `runs_df` Word) pour faire de la traduction granulaire au niveau span. Aujourd'hui placeholder vide. À construire via `page.get_text("dict")` PyMuPDF, en coordination avec Sylvère qui touche `parsing/pdf/toc/`.